# Missouri Botanical Garden Plant Finder Scraper

Scrapes Bloom Time, Bloom Description, and Flower fields for a list of scientific names.

Flow per name:
1. GET search page → grab ASP.NET viewstate tokens
2. POST search form with `BasicListView` selected and the name in the search box
3. If a result page comes back: scan the result `<a>` tags for a case-insensitive exact match
4. If a detail page comes back directly (single-hit redirect): scrape it
5. Pull the three fields; missing field → `N/a`

In [21]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
import geopandas as gpd
BASE_URL = "https://www.missouribotanicalgarden.org"
SEARCH_URL = f"{BASE_URL}/plantfinder/plantfindersearch.aspx"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://www.missouribotanicalgarden.org/plantfinder/plantfindersearch.aspx",
    "Upgrade-Insecure-Requests": "1",
}

REQUEST_DELAY = 1.0  # seconds between requests — be polite

## Helpers

In [22]:
def normalize(text: str) -> str:
    """Collapse whitespace and lowercase for matching."""
    return re.sub(r"\s+", " ", text).strip().lower()


def get_form_state(session: requests.Session) -> dict:
    """Fetch the search page and return ASP.NET hidden form fields."""
    resp = session.get(SEARCH_URL, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    state = {}
    for field in ("__VIEWSTATE", "__VIEWSTATEGENERATOR", "__EVENTVALIDATION"):
        tag = soup.find("input", {"name": field})
        state[field] = tag["value"] if tag else ""
    return state


def submit_search(session: requests.Session, name: str, state: dict) -> requests.Response:
    """POST the search form with BasicListView selected."""
    form_data = {
        "__EVENTTARGET": "",
        "__EVENTARGUMENT": "",
        **state,
        "ctl00$MainContentPlaceHolder$BasicSearchCtrl$BasicSearchType": "BasicListView",
        "ctl00$MainContentPlaceHolder$BasicSearchCtrl$CommonNameSearch": name,
        "ctl00$MainContentPlaceHolder$BasicSearchCtrl$BasicSearchButton": "Search",
    }
    resp = session.post(SEARCH_URL, data=form_data, headers=HEADERS, timeout=30, allow_redirects=True)
    resp.raise_for_status()
    return resp

In [23]:
def is_detail_page(soup: BeautifulSoup) -> bool:
    """Detail pages have the Common Name row; result lists don't."""
    return soup.find("div", id="MainContentPlaceHolder_CommonNameRow") is not None


def find_exact_match_url(soup: BeautifulSoup, name: str) -> str | None:
    """Scan list-view results for a case-insensitive exact text match.
    Matches against the <a>'s visible text only (excluding the common-name suffix
    that lives outside the anchor)."""
    target = normalize(name)
    # The results container has id containing 'SearchResults_pnlList'. The dnn_ctrXXXX
    # prefix can vary across deploys, so match by suffix.
    container = soup.find("div", id=re.compile(r"SearchResults_pnlList$"))
    if container is None:
        return None
    for link in container.find_all("a", href=True):
        link_text = normalize(link.get_text(" ", strip=True))
        if link_text == target:
            return urljoin(BASE_URL, link["href"])
    return None


def extract_field(soup: BeautifulSoup, row_id: str, label: str) -> str:
    row = soup.find("div", id=row_id)
    if row is None:
        return "N/a"
    text = row.get_text(" ", strip=True)
    # Strip the leading label, e.g. 'Bloom Time:'
    if text.lower().startswith(label.lower()):
        text = text[len(label):]
    return text.strip(" :") or "N/a"


def extract_unlabeled_row(soup: BeautifulSoup, label: str) -> str:
    """Some rows (Flower, Leaf, Attracts, Suggested Use, ...) have no id —
    they're plain div.row elements. Find by leading text label."""
    needle = label.lower() + ":"
    for row in soup.find_all("div", class_="row"):
        text = row.get_text(" ", strip=True)
        if text.lower().startswith(needle):
            return text.split(":", 1)[1].strip() or "N/a"
    return "N/a"


def scrape_detail(soup: BeautifulSoup) -> dict:
    return {
        "Bloom Time": extract_field(soup, "MainContentPlaceHolder_BloomTimeRow", "Bloom Time:"),
        "Bloom Description": extract_field(soup, "MainContentPlaceHolder_ColorTextRow", "Bloom Description:"),
        "Flower": extract_unlabeled_row(soup, "Flower"),
        "Leaf": extract_unlabeled_row(soup, "Leaf"),
    }

In [24]:
def lookup_plant(session: requests.Session, name: str) -> dict:
    """Run the full search → match → scrape pipeline for one name."""
    result = {
        "Scientific Name": name,
        "Bloom Time": "N/a",
        "Bloom Description": "N/a",
        "Flower": "N/a",
        "Leaf": "N/a",
        "Status": "",
        "Detail URL": "",
    }
    try:
        state = get_form_state(session)
        resp = submit_search(session, name, state)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Case 1: search jumped straight to a detail page (single match)
        if is_detail_page(soup):
            result.update(scrape_detail(soup))
            result["Status"] = "matched (single)"
            result["Detail URL"] = resp.url
            return result

        # Case 2: list view — look for exact match
        detail_url = find_exact_match_url(soup, name)
        if detail_url is None:
            result["Status"] = "no exact match"
            return result

        time.sleep(REQUEST_DELAY)
        detail_resp = session.get(detail_url, headers=HEADERS, timeout=30)
        detail_resp.raise_for_status()
        detail_soup = BeautifulSoup(detail_resp.text, "html.parser")
        result.update(scrape_detail(detail_soup))
        result["Status"] = "matched"
        result["Detail URL"] = detail_url
        return result

    except requests.RequestException as e:
        result["Status"] = f"request error: {e}"
        return result
    except Exception as e:
        result["Status"] = f"error: {e}"
        return result

In [25]:
from difflib import SequenceMatcher


def similarity(a: str, b: str) -> float:
    """Ratio in [0, 1]. Empty strings → 0."""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def parse_result_row(div) -> dict | None:
    """Pull scientific name (link text), common name (trailing text), and URL
    from a single result row div."""
    link = div.find("a", href=True)
    if link is None:
        return None
    sci_raw = link.get_text(" ", strip=True)
    full = div.get_text(" ", strip=True)
    # Everything after the link text, with the ' - ' separator stripped
    common_raw = full[len(sci_raw):].lstrip(" -\u2013\u2014").strip()
    return {
        "scientific": normalize(sci_raw),
        "common": normalize(common_raw),
        "url": urljoin(BASE_URL, link["href"]),
    }


def find_best_match(soup, scientific_name: str, common_name: str | None) -> tuple[str | None, str]:
    """Return (detail_url, status). Applies single-result rule first, then fuzzy scoring."""
    container = soup.find("div", id=re.compile(r"SearchResults_pnlList$"))
    if container is None:
        return None, "no results container"

    rows = [parse_result_row(d) for d in container.find_all("div", recursive=False)]
    rows = [r for r in rows if r is not None]

    if not rows:
        return None, "no results"

    # Rule 1: single result — just take it
    if len(rows) == 1:
        return rows[0]["url"], "single result"

    # Rule 2: fuzzy score on scientific + common
    q_sci = normalize(scientific_name)
    q_com = normalize(common_name) if common_name else ""

    scored = []
    for r in rows:
        s_sci = similarity(q_sci, r["scientific"])
        s_com = similarity(q_com, r["common"])
        scored.append((s_sci + s_com, s_sci, s_com, r))
    scored.sort(key=lambda x: x[0], reverse=True)
    total, s_sci, s_com, best = scored[0]
    return best["url"], f"fuzzy (sci={s_sci:.2f}, com={s_com:.2f})"

In [26]:
def lookup_plant_fuzzy(session: requests.Session, scientific_name: str, common_name: str | None = None) -> dict:
    """Second-pass lookup: single-result rule first, then fuzzy match using common name."""
    result = {
        "Scientific Name": scientific_name,
        "Bloom Time": "N/a", "Bloom Description": "N/a", "Flower": "N/a", "Leaf": "N/a",
        "Status": "", "Detail URL": "",
    }
    try:
        state = get_form_state(session)
        resp = submit_search(session, scientific_name, state)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Single-result auto-redirect to detail page
        if is_detail_page(soup):
            result.update(scrape_detail(soup))
            result["Status"] = "matched (single redirect)"
            result["Detail URL"] = resp.url
            return result

        detail_url, status = find_best_match(soup, scientific_name, common_name)
        if detail_url is None:
            result["Status"] = status
            return result

        time.sleep(REQUEST_DELAY)
        detail_resp = session.get(detail_url, headers=HEADERS, timeout=30)
        detail_resp.raise_for_status()
        detail_soup = BeautifulSoup(detail_resp.text, "html.parser")
        result.update(scrape_detail(detail_soup))
        result["Status"] = status
        result["Detail URL"] = detail_url
        return result

    except Exception as e:
        result["Status"] = f"error: {e}"
        return result

## Run

Drop your scientific names into `names_to_lookup` (or load from a CSV).

In [37]:
trees = gpd.read_file("../data/processed/philly_trees.geojson")
trees["scientific_name"] = trees["scientific_name"].str.lower()
trees["common_name"] = trees["common_name"].str.lower()

In [45]:
species = trees.drop_duplicates(subset="scientific_name")

names_to_lookup = list(species["scientific_name"])
print(len(names_to_lookup))

# Or, from a file:
# names_to_lookup = pd.read_csv("species_list.csv")["scientific_name"].dropna().tolist()

251


In [11]:
rows = []
with requests.Session() as session:
    for i, name in enumerate(names_to_lookup, 1):
        print(f"[{i}/{len(names_to_lookup)}] {name}")
        row = lookup_plant(session, name)
        print(f"    → {row['Status']}")
        rows.append(row)
        time.sleep(REQUEST_DELAY)

df = pd.DataFrame(rows, columns=[
    "Scientific Name", "Bloom Time", "Bloom Description", "Flower", "Leaf", "Status", "Detail URL"
])
df

[1/251] ginkgo biloba
    → matched
[2/251] acer palmatum
    → matched
[3/251] acer pseudoplatanus
    → matched
[4/251] ilex opaca
    → matched
[5/251] koelreuteria paniculata
    → matched
[6/251] cornus kousa
    → matched
[7/251] amelanchier species
    → no exact match
[8/251] metasequoia glyptostroboides
    → matched
[9/251] acer rubrum
    → matched
[10/251] liriodendron tulipifera
    → matched
[11/251] fraxinus americana
    → matched
[12/251] betula nigra
    → matched
[13/251] taxodium distichum
    → no exact match
[14/251] liquidambar styraciflua
    → matched
[15/251] quercus palustris
    → matched
[16/251] pinus strobus
    → matched
[17/251] prunus sargentii
    → matched
[18/251] gleditsia triacanthos inermis
    → no exact match
[19/251] robinia pseudoacacia
    → matched
[20/251] prunus species
    → no exact match
[21/251] maackia amurensis
    → matched
[22/251] syringa reticulata
    → matched
[23/251] parrotia persica
    → matched
[24/251] styrax japonicus
 

,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL
0,ginkgo biloba,April,Green,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
1,acer palmatum,April,Reddish-purple,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
2,acer pseudoplatanus,May,Yellow green,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...
3,ilex opaca,May,Creamy white,Insignificant,Evergreen,matched,https://www.missouribotanicalgarden.org/PlantF...
4,koelreuteria paniculata,June to July,Yellow,Showy,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
...,...,...,...,...,...,...,...
246,sorbus aucuparia,May,White,Showy,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
247,sorbus americana,May,White,Showy,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...
248,acer henryii,N/a,N/a,N/a,N/a,no exact match,
249,ulmus alata,March to April,Reddish green,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...


In [30]:
#df.to_csv("mobot_plant_data.csv", index=False)
print(f"Saved {len(df)} rows → mobot_plant_data.csv")

Saved 251 rows → mobot_plant_data.csv


## Second pass: single-result + fuzzy fallback

After the first pass, gather rows where `Status == 'no exact match'`. Build a `{scientific_name: common_name}` dict for those and run them through `lookup_plant_fuzzy`. The rules:

1. If the search returns a single result, take it (no name comparison needed — common case is the base species existing only as a named cultivar, e.g. `pinus virginiana` → `Pinus virginiana 'Wate's Golden'`).
2. If multiple results, score each result against your scientific name *and* the common name you provide, pick the highest combined similarity. The common name is what discriminates between cultivars.

Uses `difflib.SequenceMatcher` from stdlib — no extra dependency.

### Run the fallback on failed names

After your first-pass `df` is built, identify the failures, fill in the `common_names` dict, and run the fallback. Results get merged back into `df`.

In [46]:
misses = df.loc[df["Status"] != "matched"]
specific_misses = misses.loc[~misses["Scientific Name"].str.contains("species")]
specific_misses


,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL
12,taxodium distichum,N/a,N/a,N/a,N/a,no exact match,
17,gleditsia triacanthos inermis,N/a,N/a,N/a,N/a,no exact match,
30,quercus laevis,N/a,N/a,N/a,N/a,no exact match,
32,platanus acerifolia,N/a,N/a,N/a,N/a,no exact match,
39,aesculus octandra,N/a,N/a,N/a,N/a,no exact match,
47,pyrus calleryana,N/a,N/a,N/a,N/a,no exact match,
56,aesculus carnea,N/a,N/a,N/a,N/a,no exact match,
63,acer freemanii,N/a,N/a,N/a,N/a,no exact match,
67,magnolia soulangiana,N/a,N/a,N/a,N/a,no exact match,
74,cedrella sinensis,N/a,N/a,N/a,N/a,no exact match,


In [48]:
# create list of missed scientific names
missed_s_names = list(specific_misses["Scientific Name"])
missed_s_names
# take slice of tree databse with the only species info for missed ones
missed_species = species[species["scientific_name"].isin(missed_s_names)]
missed_species
name_dict = dict(zip(missed_species["scientific_name"], missed_species["common_name"]))



In [49]:
# 2. Fill in the common name for each failed scientific name.
#    Common name is only needed when there are multiple results — for single-result
#    species you can leave the value as None or empty.
common_names = name_dict

# 3. Run the fallback
fallback_rows = []
with requests.Session() as session:
    for i, (sci_name, com_name) in enumerate(common_names.items(), 1):
        print(f"[{i}/{len(common_names)}] {sci_name}  (common: {com_name!r})")
        row = lookup_plant_fuzzy(session, sci_name, com_name)
        print(f"    → {row['Status']}")
        fallback_rows.append(row)
        time.sleep(REQUEST_DELAY)

fallback_df = pd.DataFrame(fallback_rows, columns=df.columns)
fallback_df

[1/51] taxodium distichum  (common: 'baldcypress')
    → fuzzy (sci=0.78, com=0.96)
[2/51] gleditsia triacanthos inermis  (common: 'thornless honeylocust')
    → no results container
[3/51] quercus laevis  (common: 'turkey oak')
    → no results container
[4/51] platanus acerifolia  (common: 'london planetree')
    → no results container
[5/51] aesculus octandra  (common: 'yellow buckeye')
    → no results container
[6/51] pyrus calleryana  (common: 'callery pear')
    → fuzzy (sci=0.76, com=1.00)
[7/51] aesculus carnea  (common: 'red horsechestnut')
    → no results container
[8/51] acer freemanii  (common: 'armstrong maple')
    → no results container
[9/51] magnolia soulangiana  (common: 'saucer magnolia')
    → no results container
[10/51] cedrella sinensis  (common: 'chinese toon')
    → no results container
[11/51] acer nigrum  (common: 'black maple')
    → no results container
[12/51] prunus persica  (common: 'peach')
    → fuzzy (sci=0.74, com=1.00)
[13/51] shrub shrub  (common

,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL
0,taxodium distichum,Non-flowering,Non-flowering,N/a,Good Fall,"fuzzy (sci=0.78, com=0.96)",https://www.missouribotanicalgarden.org/PlantF...
1,gleditsia triacanthos inermis,N/a,N/a,N/a,N/a,no results container,
2,quercus laevis,N/a,N/a,N/a,N/a,no results container,
3,platanus acerifolia,N/a,N/a,N/a,N/a,no results container,
4,aesculus octandra,N/a,N/a,N/a,N/a,no results container,
5,pyrus calleryana,April,White,Showy,Good Fall,"fuzzy (sci=0.76, com=1.00)",https://www.missouribotanicalgarden.org/PlantF...
6,aesculus carnea,N/a,N/a,N/a,N/a,no results container,
7,acer freemanii,N/a,N/a,N/a,N/a,no results container,
8,magnolia soulangiana,N/a,N/a,N/a,N/a,no results container,
9,cedrella sinensis,N/a,N/a,N/a,N/a,no results container,


In [50]:
# 4. Merge fallback results back into the main df, overwriting the failed rows
df = df.set_index("Scientific Name")
fb = fallback_df.set_index("Scientific Name")
df.update(fb)  # overwrites only matching index values
df = df.reset_index()
df

,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL
0,ginkgo biloba,April,Green,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
1,acer palmatum,April,Reddish-purple,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
2,acer pseudoplatanus,May,Yellow green,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...
3,ilex opaca,May,Creamy white,Insignificant,Evergreen,matched,https://www.missouribotanicalgarden.org/PlantF...
4,koelreuteria paniculata,June to July,Yellow,Showy,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
...,...,...,...,...,...,...,...
246,sorbus aucuparia,May,White,Showy,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...
247,sorbus americana,May,White,Showy,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...
248,acer henryii,N/a,N/a,N/a,N/a,no results container,
249,ulmus alata,March to April,Reddish green,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...


In [53]:
df.to_csv("mobot_plant_data.csv", index=False)
print(f"Saved final {len(df)} rows → mobot_plant_data.csv")

Saved final 251 rows → mobot_plant_data.csv


### Debugging tips

- If everything comes back `no exact match`, inspect a response: `print(BeautifulSoup(resp.text, 'html.parser').find('div', id=re.compile(r'SearchResults_pnlList$')))` to see what the list actually looks like. Cultivar names with curly vs straight quotes (`'` vs `'`) trip exact match — normalize your input list.
- If you get blocked / 403s, raise `REQUEST_DELAY` to 2–3 seconds.
- The search box is labeled `CommonNameSearch` in the HTML, but it accepts scientific names too (it's a unified search). No changes needed.